# Correr Test 1 ($T_n$, Schuster-Barker)

Este notebook permite ejecutar el estudio Monte Carlo del Test 1 con parámetros configurables. Las salidas se guardan en `results/data/` y `results/figures/`.

**Modo recomendado:** `parallel` con `quick=True` para verificación rápida (~1 min), `quick=False` para corrida completa (~15-20 min con 12 cores).

Para más detalle ver `documentacion/documentacion_test1.pdf`.

In [ ]:
# --- Setup: imports y path al repo ---
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.distributions import default_h0_specs, default_ha_specs
from src.simulation import SimConfig, summarize

print(f'Working dir: {Path.cwd()}')
print(f'CPU cores disponibles: {os.cpu_count()}')

## Parámetros

Edita esta celda para cambiar la configuración del estudio.

In [ ]:
# --- Parámetros de la simulación ---
quick = True              # True: B=199, R=80, n=(20,40,80). False: full (B=500, R=300, n=(20,40,80,160))
use_parallel = True       # True: paraleliza con ProcessPoolExecutor. False: secuencial.

if quick:
    config = SimConfig(
        sample_sizes=(20, 40, 80),
        estimators=('argmin', 'median', 'trimmed'),
        B=199, R=80, alpha=0.05, seed=2026,
    )
    suffix = '_quick'
else:
    config = SimConfig(
        sample_sizes=(20, 40, 80, 160),
        estimators=('argmin', 'median', 'trimmed'),
        B=500, R=300, alpha=0.05, seed=2026,
    )
    suffix = ''

specs = default_h0_specs() + default_ha_specs()
print(f'Distribuciones ({len(specs)}): {[s.name for s in specs]}')
print(f'Tamaños n: {config.sample_sizes}')
print(f'Estimadores: {config.estimators}')
print(f'B={config.B}, R={config.R}')
total = len(specs) * len(config.sample_sizes) * len(config.estimators) * config.R
print(f'Tareas totales: {total:,}')

## Ejecutar la simulación

In [ ]:
# --- Run ---
if use_parallel:
    from src.simulation_parallel import run_simulation_parallel
    n_workers = max(1, (os.cpu_count() or 2) - 1)
    df = run_simulation_parallel(specs, config, n_workers=n_workers)
else:
    from src.simulation import run_simulation
    df = run_simulation(specs, config)

summary = summarize(df, alpha=config.alpha)
print(f'\nFilas raw: {len(df):,}  |  Filas summary: {len(summary)}')

## Guardar resultados

In [ ]:
data_dir = ROOT / 'results' / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
raw_csv = data_dir / f'tn_simulation_raw{suffix}.csv'
sum_csv = data_dir / f'tn_simulation_summary{suffix}.csv'
df.to_csv(raw_csv, index=False)
summary.to_csv(sum_csv, index=False)
print(f'Guardado:\n  {raw_csv}\n  {sum_csv}')

## Resumen rápido

In [ ]:
import pandas as pd

print('Tasas de rechazo (filas: dist x estimador, columnas: n):')
pv = summary.pivot_table(index=['dist','estimator'], columns='n', values='reject_rate').round(3)
pv

In [ ]:
print('Tiempo medio por test (s):')
summary.pivot_table(index=['dist','estimator'], columns='n', values='mean_time_s').round(3)

Para generar gráficas usar el notebook `03_plots_test1.ipynb`.